In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
# 한글 폰트 설정
plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)
import matplotlib.font_manager as fm

# 폰트 경로 설정
font_path = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'  # MacOS
# font_path = 'C:/Windows/Fonts/malgun.ttf'  # Windows

# 폰트 추가
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'AppleGothic'  # MacOS
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 경고 메시지 무시 설정
import warnings
warnings.filterwarnings('ignore')

# pandas의 SettingWithCopyWarning 무시
pd.options.mode.chained_assignment = None


In [2]:
df = pd.read_excel('./data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])



In [3]:
# df.head()
# df.describe()
# df.info()
df = df.iloc[:,1:]
df.columns = df.columns.str.strip()
# df.columns

In [4]:
df[['CC','약','장치','습관','찜질','마사지, 스트레칭','PI']] =df[['CC','약','장치','습관','찜질','마사지, 스트레칭','PI']].fillna('')

In [5]:
api_keys = pd.read_csv('../info.csv').iloc[0,1]

In [ ]:
df.head()

In [ ]:
import pandas as pd
import anthropic
from typing import List, Dict
import json
from tqdm import tqdm

class ClaudeClassifier:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        
    def classify_jjimjil(self, texts: List[str], batch_size: int = 20) -> List[Dict]:
        """찜질 관련 텍스트 분류"""
        results = []
        
        # 배치 단위로 처리
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i + batch_size]
            batch_results = self._process_batch(batch_texts)
            results.extend(batch_results)
            
        return results
    
    def _process_batch(self, texts: List[str]) -> List[Dict]:
        """배치 단위 텍스트 처리"""
        
        # 텍스트 목록 포맷팅
        formatted_texts = "\n".join([f"- {text}" for text in texts])
        
        prompt = f"""다음 찜질 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

각 텍스트에 대해 다음 정보를 추출해주세요:
1. status: 찜질 시행 여부 (0: 미시행, 1: 시행)
2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
4. method: 찜질 방법 ('hot': 온찜질, 'cold': 냉찜질, 'both': 둘 다, null: 불명확)

텍스트 목록:
{formatted_texts}

다음과 같은 JSON 형식으로 응답해주세요:
[
    {{
        "text": "원본 텍스트",
        "classification": {{
            "status": 0 또는 1,
            "frequency": "high/medium/low 중 하나 또는 null",
            "duration": 숫자 또는 null,
            "method": "hot/cold/both 중 하나 또는 null"
        }}
    }}
]"""
        
        # Claude API 호출
        response = self.client.messages.create(
            model="claude-3-5-haiku-20241022",
            max_tokens=4096,
            temperature=0,
            system="JSON 형식의 분류 결과만 반환하세요.",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )
        
        try:
            # JSON 파싱
            return json.loads(response.content[0].text)
        except json.JSONDecodeError as e:
            print(f"JSON 파싱 에러: {e}")
            print(f"원본 응답: {response.content}")
            return [{"text": text, "classification": {"status": None, "frequency": None, "duration": None, "method": None}} for text in texts]

def process_dataframe(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """데이터프레임 전체 처리"""
    classifier = ClaudeClassifier(api_key)
    
    # 찜질 열 분류 (공백 포함된 컬럼명 사용)
    texts = df['찜질'].fillna('').tolist()
    results = classifier.classify_jjimjil(texts)
    
    # 결과를 데이터프레임에 추가
    df['찜질_상태'] = [r['classification']['status'] for r in results]
    df['찜질_빈도'] = [r['classification']['frequency'] for r in results]
    df['찜질_시간'] = [r['classification']['duration'] for r in results]
    df['찜질_방법'] = [r['classification']['method'] for r in results]
    
    return df

# 사용 예시
if __name__ == "__main__":
    # CSV 파일 읽기
    # df = pd.read_csv('test.csv')

    # 전처리 수행
    processed_df = process_dataframe(df, api_keys)
    
    # 결과 확인
    print("\n=== 전처리 결과 통계 ===")
    print("\n1. 찜질 상태 분포:")
    print(processed_df['찜질_상태'].value_counts(dropna=False))
    
    print("\n2. 찜질 빈도 분포:")
    print(processed_df['찜질_빈도'].value_counts(dropna=False))
    
    print("\n3. 찜질 방법 분포:")
    print(processed_df['찜질_방법'].value_counts(dropna=False))
    
    # 샘플 데이터 출력
    print("\n4. 처리 결과 샘플:")
    sample_cols = ['찜질', '찜질_상태', '찜질_빈도', '찜질_시간', '찜질_방법']
    print(processed_df[sample_cols].head())

In [ ]:
import pandas as pd
import anthropic
from typing import List, Dict
import json
import asyncio
import nest_asyncio
nest_asyncio.apply()
import re
from tqdm.asyncio import tqdm_asyncio

class ClaudeClassifier:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        
    async def classify_jjimjil(self, texts: List[str], batch_size: int = 50) -> List[Dict]:
        """찜질 관련 텍스트 분류 (비동기)"""
        results = []
        
        # 공란이 아닌 텍스트만 처리하되, 인덱스 정보 유지
        valid_texts_with_idx = [(idx, text) for idx, text in enumerate(texts) if pd.notna(text) and text.strip()]
        if not valid_texts_with_idx:
            return []
            
        valid_indices, valid_texts = zip(*valid_texts_with_idx)
        
        batches = [valid_texts[i:i + batch_size] for i in range(0, len(valid_texts), batch_size)]
        batch_indices = [valid_indices[i:i + batch_size] for i in range(0, len(valid_indices), batch_size)]
        
        semaphore = asyncio.Semaphore(5)
        
        async def process_with_semaphore(batch, indices):
            async with semaphore:
                batch_results = await self._process_batch(batch)
                # 인덱스 정보 추가
                for result, idx in zip(batch_results, indices):
                    result['original_index'] = idx
                return batch_results
        
        async for batch_results in tqdm_asyncio(
            self._process_batches(zip(batches, batch_indices), process_with_semaphore),
            total=len(batches),
            desc="Processing batches"
        ):
            results.extend(batch_results)
            
        # 원본 인덱스로 정렬
        results.sort(key=lambda x: x['original_index'])
        return results
    
    async def _process_batches(self, batch_items, process_func):
        for batch, indices in batch_items:
            result = await process_func(batch, indices)
            yield result
    
    async def _process_batch(self, texts: List[str]) -> List[Dict]:
        """배치 단위 텍스트 처리 (비동기)"""
        formatted_texts = "\n".join(f"- {text}" for text in texts)
        
        prompt = f"""다음 찜질 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        
각 텍스트에 대해 다음 정보를 추출해주세요:
1. status: 찜질 시행 여부 (0: 미시행, 1: 시행)
2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
4. method: 찜질 방법 ('hot': 온찜질, 'cold': 냉찜질, 'both': 둘 다, null: 불명확)

텍스트 목록:
{formatted_texts}"""

        try:
            response = await asyncio.to_thread(
                self.client.messages.create,
                model="claude-3-5-haiku-20241022",
                max_tokens=4096,
                temperature=0,
                system="JSON 형식의 분류 결과만 반환하세요.",
                messages=[{"role": "user", "content": prompt}]
            )
            
            content = response.content[0].text
            content = re.sub(r'```json\n|\n```', '', content)
            
            return json.loads(content)
            
        except Exception as e:
            print(f"Error processing batch: {e}")
            return [{"text": text, "classification": {
                "status": None, "frequency": None,
                "duration": None, "method": None
            }} for text in texts]

async def process_dataframe_async(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """데이터프레임 비동기 처리"""
    classifier = ClaudeClassifier(api_key)
    
    # 원본 인덱스 보존
    texts = df['찜질'].tolist()
    results = await classifier.classify_jjimjil(texts)
    
    # 결과 데이터프레임 생성
    result_df = pd.DataFrame({
        'index': range(len(df)),  # 원본 인덱스 유지
        '찜질_상태': [None] * len(df),
        '찜질_빈도': [None] * len(df),
        '찜질_시간': [None] * len(df),
        '찜질_방법': [None] * len(df)
    })
    
    # 처리된 결과만 업데이트
    for result in results:
        idx = result['original_index']
        result_df.loc[idx, '찜질_상태'] = result['classification']['status']
        result_df.loc[idx, '찜질_빈도'] = result['classification']['frequency']
        result_df.loc[idx, '찜질_시간'] = result['classification']['duration']
        result_df.loc[idx, '찜질_방법'] = result['classification']['method']
    
    # 원본 데이터프레임과 병합
    result_df.set_index('index', inplace=True)
    df = df.join(result_df)
    
    return df

# 사용 예시
if __name__ == "__main__":
    # df = pd.read_csv('test.csv')
    processed_df = asyncio.run(process_dataframe_async(df, api_keys))
    
    print("\n=== 전처리 결과 통계 ===")
    print("\n1. 찜질 상태 분포:")
    print(processed_df['찜질_상태'].value_counts(dropna=False))
    
    print("\n2. 찜질 빈도 분포:")
    print(processed_df['찜질_빈도'].value_counts(dropna=False))
    
    print("\n3. 찜질 방법 분포:")
    print(processed_df['찜질_방법'].value_counts(dropna=False))
    
    print("\n4. 처리 결과 샘플:")
    sample_cols = ['찜질', '찜질_상태', '찜질_빈도', '찜질_시간', '찜질_방법']
    print(processed_df[sample_cols].head())

In [ ]:
import pandas as pd
import anthropic
from typing import List, Dict
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio

nest_asyncio.apply()

class ClaudeClassifier:
   def __init__(self, api_key: str):
       self.client = anthropic.Anthropic(api_key=api_key)
       
   async def classify_jjimjil(self, texts: List[str], batch_size: int = 50) -> List[Dict]:
       """찜질 관련 텍스트 분류 (비동기)"""
       results = []
       batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
       
       semaphore = asyncio.Semaphore(5)
       
       async def process_with_semaphore(batch):
           async with semaphore:
               return await self._process_batch(batch)
       
       # 비동기로 배치 처리
       for batch in tqdm_asyncio(batches, desc="Processing batches"):
           batch_results = await process_with_semaphore(batch)
           results.extend(batch_results)
           
       return results
   
   async def _process_batch(self, texts: List[str]) -> List[Dict]:
       """배치 단위 텍스트 처리"""
       # 빈 텍스트 필터링
       texts = [text for text in texts if pd.notna(text) and text.strip()]
       if not texts:
           return []
           
       formatted_texts = "\n".join(f"- {text}" for text in texts)
       
       prompt = f"""다음 찜질 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
   
각 텍스트에 대해 다음 정보를 추출해주세요:
1. status: 찜질 시행 여부 (0: 미시행, 1: 시행)
2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
4. method: 찜질 방법 ('hot': 온찜질, 'cold': 냉찜질, 'both': 둘 다, null: 불명확)

텍스트 목록:
{formatted_texts}

다음과 같은 JSON 형식으로 응답해주세요:
[
   {{
       "text": "원본 텍스트",
       "classification": {{
           "status": 0 또는 1,
           "frequency": "high/medium/low 중 하나 또는 null",
           "duration": 숫자 또는 null,
           "method": "hot/cold/both 중 하나 또는 null"
       }}
   }}
]"""

       try:
           response = await asyncio.to_thread(
               self.client.messages.create,
               model="claude-3-5-haiku-20241022",
               max_tokens=4096,
               temperature=0,
               system="JSON 형식의 분류 결과만 반환하세요.",
               messages=[{"role": "user", "content": prompt}]
           )
           
           content = response.content[0].text
           content = re.sub(r'```json\n|\n```', '', content)
           
           parsed_results = json.loads(content)
           print(f"Parsed results: {parsed_results[:2]}")  # 디버깅용
           
           # 결과 구조 확인 및 보정
           corrected_results = []
           for result in parsed_results:
               if 'classification' not in result:
                   corrected_result = {
                       'text': result.get('text', ''),
                       'classification': {
                           'status': result.get('status', None),
                           'frequency': result.get('frequency', None),
                           'duration': result.get('duration', None),
                           'method': result.get('method', None)
                       }
                   }
                   corrected_results.append(corrected_result)
               else:
                   corrected_results.append(result)
                   
           return corrected_results
               
       except Exception as e:
           print(f"Error processing batch: {e}")
           print(f"원본 응답: {response.content if 'response' in locals() else 'No response'}")
           return [{"text": text, "classification": {
               "status": None, "frequency": None,
               "duration": None, "method": None
           }} for text in texts]

async def process_dataframe_async(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
   """데이터프레임 비동기 처리"""
   classifier = ClaudeClassifier(api_key)
   
   # 공란이 아닌 텍스트만 처리
   mask = df['찜질'].notna() & df['찜질'].str.strip().astype(bool)
   texts = df.loc[mask, '찜질'].tolist()
   
   # 분류 결과 얻기
   results = await classifier.classify_jjimjil(texts)
   
   # 결과 데이터프레임 생성
   result_df = pd.DataFrame(columns=['찜질_상태', '찜질_빈도', '찜질_시간', '찜질_방법'])
   
   # 유효한 텍스트에 대한 결과만 추가
   for text, result in zip(texts, results):
       result_df = pd.concat([result_df, pd.DataFrame({
           '찜질_상태': [result['classification']['status']],
           '찜질_빈도': [result['classification']['frequency']],
           '찜질_시간': [result['classification']['duration']],
           '찜질_방법': [result['classification']['method']]
       })], ignore_index=True)
   
   # 기존 데이터프레임에 새 컬럼 추가 (아직 없는 경우)
   new_columns = ['찜질_상태', '찜질_빈도', '찜질_시간', '찜질_방법']
   for col in new_columns:
       if col not in df.columns:
           df[col] = None
   
   # 원본 데이터프레임에 결과 병합
   df.loc[mask, '찜질_상태'] = result_df['찜질_상태']
   df.loc[mask, '찜질_빈도'] = result_df['찜질_빈도']
   df.loc[mask, '찜질_시간'] = result_df['찜질_시간']
   df.loc[mask, '찜질_방법'] = result_df['찜질_방법']
   
   return df

if __name__ == "__main__":
   # df = pd.read_csv('test.csv')
   loop = asyncio.get_event_loop()
   processed_df = loop.run_until_complete(process_dataframe_async(df.head(100), api_keys))
   
   print("\n=== 전처리 결과 통계 ===")
   print("\n1. 찜질 상태 분포:")
   print(processed_df['찜질_상태'].value_counts(dropna=False))
   
   print("\n2. 찜질 빈도 분포:")
   print(processed_df['찜질_빈도'].value_counts(dropna=False))
   
   print("\n3. 찜질 방법 분포:")
   print(processed_df['찜질_방법'].value_counts(dropna=False))
   
   print("\n4. 처리 결과 샘플:")
   sample_cols = ['찜질', '찜질_상태', '찜질_빈도', '찜질_시간', '찜질_방법']
   print(processed_df[sample_cols].head())

In [ ]:
import pandas as pd
import anthropic
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime

nest_asyncio.apply()

class MedicalTextClassifier:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(5)  # Create semaphore at initialization
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
            'PI': self._classify_present_illness
        }
    
    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 열 처리"""
        for column in self.classifiers.keys():
            if column in df.columns:
                print(f"\nProcessing column: {column}")
                try:
                    mask = df[column].notna() & df[column].str.strip().astype(bool)
                    texts = df.loc[mask, column].tolist()
                    results = await self.process_column(column, texts)
                    
                    if results:  # Only process if we have results
                        # 결과를 데이터프레임에 병합
                        result_df = pd.json_normalize(results)
                        for new_col in result_df.columns:
                            col_name = f"{column}_{new_col}"
                            df.loc[mask, col_name] = result_df[new_col]
                        
                except Exception as e:
                    print(f"Error processing column {column}: {str(e)}")
                    
        return df

    async def process_column(self, column_name: str, texts: List[str], batch_size: int = 50) -> List[Dict]:
        """특정 열의 텍스트들을 분류"""
        if column_name not in self.classifiers:
            raise ValueError(f"Unknown column: {column_name}")
            
        classifier = self.classifiers[column_name]
        return await self._process_batches(texts, classifier, batch_size)
    
    async def _process_batches(self, texts: List[str], classifier_func, batch_size: int) -> List[Dict]:
        """배치 처리 공통 로직"""
        results = []
        texts = [text for text in texts if pd.notna(text) and str(text).strip()]
        
        if not texts:  # Return empty list if no valid texts
            return []
            
        batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
        
        async with self.semaphore:  # Use instance semaphore
            for batch in tqdm_asyncio(batches, desc=f"Processing {classifier_func.__name__}"):
                try:
                    batch_results = await classifier_func(batch)
                    if batch_results:
                        results.extend(batch_results)
                except Exception as e:
                    print(f"Error processing batch: {str(e)}")
                    continue
                
        return results

    async def _make_api_call(self, prompt: str) -> List[Dict]:
        """API 호출 공통 로직"""
        try:
            response = await asyncio.to_thread(
                self.client.messages.create,
                model="claude-3-5-haiku-20241022",
                max_tokens=4096,
                temperature=0,
                system="JSON 형식의 분류 결과만 반환하세요.",
                messages=[{"role": "user", "content": prompt}]
            )
            
            content = response.content[0].text
            content = re.sub(r'```json\n|\n```', '', content)
            return json.loads(content)
            
        except Exception as e:
            print(f"API call error: {str(e)}")
            return []

    async def _classify_cc(self, texts: List[str]) -> List[Dict]:
        """CC(Chief Complaint) 분류"""
        prompt = f"""다음 주요 증상(CC) 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

각 텍스트에 대해 다음 정보를 추출해주세요:
1. location: 통증/증상 위치 (문자열)
2. pain_type: 통증/증상 종류 (문자열)
3. severity: 통증/증상 강도 (1-5, 없으면 null)
4. duration: 지속 기간 (명시된 경우만, 문자열)

텍스트 목록:
{texts}

다음 JSON 형식으로 응답해주세요:
[{{
    "text": "원본 텍스트",
    "location": "위치",
    "pain_type": "통증 종류",
    "severity": 숫자 또는 null,
    "duration": "기간 또는 null"
}}]"""

        return await self._make_api_call(prompt)

        return await self._make_api_call(prompt, semaphore)

    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        prompt = f"""다음 약물 복용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

각 텍스트에 대해 다음 정보를 추출해주세요:
1. medication_type: 약물 종류 (진통제/소염제/근이완제 등)
2. frequency: 복용 빈도 ('regular': 정기적, 'occasional': 간헐적, 'none': 미복용)
3. duration: 복용 기간 (명시된 경우만)
4. compliance: 복약 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

텍스트 목록:
{texts}

다음 JSON 형식으로 응답해주세요:
[{{
    "text": "원본 텍스트",
    "medication_type": "약물 종류",
    "frequency": "복용 빈도",
    "duration": "기간 또는 null",
    "compliance": "순응도"
}}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        prompt = f"""다음 장치 사용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

각 텍스트에 대해 다음 정보를 추출해주세요:
1. device_type: 장치 종류
2. usage_pattern: 사용 패턴 ('constant': 상시착용, 'partial': 부분착용, 'rare': 거의미착용)
3. duration: 사용 기간
4. compliance: 착용 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

텍스트 목록:
{texts}

다음 JSON 형식으로 응답해주세요:
[{{
    "text": "원본 텍스트",
    "device_type": "장치 종류",
    "usage_pattern": "사용 패턴",
    "duration": "기간 또는 null",
    "compliance": "순응도"
}}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        prompt = f"""다음 습관 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

각 텍스트에 대해 다음 정보를 추출해주세요:
1. habit_type: 습관 종류 (이갈이/편측성저작 등)
2. frequency: 빈도 ('high': 매일/자주, 'medium': 가끔, 'low': 거의없음)
3. awareness: 인지여부 ('aware': 인지, 'unaware': 미인지)
4. improvement: 개선여부 ('improved': 개선, 'unchanged': 유지, 'worsened': 악화)

텍스트 목록:
{texts}

다음 JSON 형식으로 응답해주세요:
[{{
    "text": "원본 텍스트",
    "habit_type": "습관 종류",
    "frequency": "빈도",
    "awareness": "인지여부",
    "improvement": "개선여부"
}}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        prompt = f"""다음 찜질 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

각 텍스트에 대해 다음 정보를 추출해주세요:
1. status: 찜질 시행 여부 (0: 미시행, 1: 시행)
2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
4. method: 찜질 방법 ('hot': 온찜질, 'cold': 냉찜질, 'both': 둘 다, null: 불명확)

텍스트 목록:
{texts}

다음 JSON 형식으로 응답해주세요:
[{{
    "text": "원본 텍스트",
    "status": 0 또는 1,
    "frequency": "빈도",
    "duration": 숫자 또는 null,
    "method": "방법"
}}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지/스트레칭 분류"""
        prompt = f"""다음 마사지/스트레칭 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

각 텍스트에 대해 다음 정보를 추출해주세요:
1. type: 종류 ('massage': 마사지, 'stretching': 스트레칭, 'both': 둘다)
2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
4. method: 방법 ('self': 자가, 'professional': 전문가, 'both': 둘다)

텍스트 목록:
{texts}

다음 JSON 형식으로 응답해주세요:
[{{
    "text": "원본 텍스트",
    "type": "종류",
    "frequency": "빈도",
    "duration": 숫자 또는 null,
    "method": "방법"
}}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """현재 질환(PI) 분류"""
        prompt = f"""다음 현재 질환(PI) 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

각 텍스트에 대해 다음 정보를 추출해주세요:
1. onset: 증상 발현 시기
2. pattern: 증상 양상 ('constant': 지속성, 'intermittent': 간헐성, 'progressive': 진행성)
3. aggravating_factors: 악화 요인 (리스트)
4. status: 현재 상태 ('improving': 호전중, 'unchanged': 유지, 'worsening': 악화)

텍스트 목록:
{texts}

다음 JSON 형식으로 응답해주세요:
[{{
    "text": "원본 텍스트",
    "onset": "발현 시기",
    "pattern": "증상 양상",
    "aggravating_factors": ["요인1", "요인2"],
    "status": "현재 상태"
}}]"""

        return await self._make_api_call(prompt, semaphore)

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """의료 데이터 전체 처리"""
    classifier = MedicalTextClassifier(api_key)
    return await classifier.process_all_columns(df)

if __name__ == "__main__":
    # 데이터 처리
    loop = asyncio.get_event_loop()
    # df = pd.read_csv('test.csv', encoding='utf-8')
    processed_df = loop.run_until_complete(process_medical_data(df.head(50), api_keys))
    
    # 결과 저장
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f'processed_medical_data_{timestamp}.csv'
    processed_df.to_csv(output_file, encoding='utf-8', index=False)
    
    # 기본 통계 출력
    print("\n=== 처리 결과 통계 ===")
    for column in processed_df.columns:
        if column.endswith(('_status', '_frequency', '_type', 'method')):
            print(f"\n{column} 분포:")
            print(processed_df[column].value_counts(dropna=False))

### 작업중


In [45]:
np.random.seed(42)
df_sample = df.sample(n=10, random_state=42)


In [ ]:
import pandas as pd
import anthropic
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime

nest_asyncio.apply()

class MedicalTextClassifier:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(5)
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
            'PI': self._classify_present_illness
        }
    
    # async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
    #     """모든 열 처리"""
    #     for column in self.classifiers.keys():
    #         if column in df.columns:
    #             print(f"\nProcessing column: {column}")
    #             try:
    #                 # 유효한 텍스트가 있는 행만 필터링
    #                 mask = df[column].notna() & df[column].str.strip().astype(bool)
    #                 if not mask.any():
    #                     continue
                    
    #                 # 텍스트와 원본 인덱스를 함께 저장
    #                 texts_with_idx = [(idx, text) for idx, text in df.loc[mask, column].items()]
    #                 texts = [text for _, text in texts_with_idx]
    #                 original_indices = [idx for idx, _ in texts_with_idx]
                    
    #                 results = await self.process_column(column, texts)
                    
    #                 if results:
    #                     # 결과를 DataFrame으로 변환
    #                     result_df = pd.json_normalize(results)
                        
    #                     # 원본 인덱스를 결과 DataFrame에 설정
    #                     result_df.index = original_indices
                        
    #                     # 새로운 컬럼 생성 및 데이터 매핑
    #                     for new_col in result_df.columns:
    #                         if new_col != 'text':  # 원본 텍스트는 제외
    #                             col_name = f"{column}_{new_col}"
    #                             # 인덱스 기반 매핑
    #                             df.loc[original_indices, col_name] = result_df[new_col]
                                
    #             except Exception as e:
    #                 print(f"Error processing column {column}: {str(e)}")
                        
    #     return df

    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 열 처리"""
        for column in self.classifiers.keys():
            if column in df.columns:
                print(f"\nProcessing column: {column}")
                try:
                    # 유효한 텍스트가 있는 행만 필터링
                    mask = df[column].notna() & df[column].str.strip().astype(bool)
                    if not mask.any():
                        continue
                    
                    # 텍스트와 원본 인덱스를 함께 저장
                    texts_with_idx = [(idx, text) for idx, text in df.loc[mask, column].items()]
                    texts = [text for _, text in texts_with_idx]
                    original_indices = [idx for idx, _ in texts_with_idx]
                    
                    results = await self.process_column(column, texts)
                    
                    if results:
                        # 결과를 DataFrame으로 변환
                        result_df = pd.DataFrame(results)
                        
                        # 새로운 컬럼 생성 및 데이터 매핑
                        for new_col in result_df.columns:
                            if new_col != 'text':  # 원본 텍스트는 제외
                                col_name = f"{column}_{new_col}"
                                # 새 컬럼 초기화
                                if col_name not in df.columns:
                                    df[col_name] = pd.NA
                                
                                # 결과 매핑 - 인덱스 기반
                                for i, idx in enumerate(original_indices):
                                    if i < len(result_df):
                                        df.at[idx, col_name] = result_df.iloc[i][new_col]
                        
                except Exception as e:
                    print(f"Error processing column {column}: {str(e)}")
                        
        return df

    async def process_column(self, column_name: str, texts: List[str], batch_size: int = 50) -> List[Dict]:
        """특정 열의 텍스트들을 분류"""
        if column_name not in self.classifiers:
            raise ValueError(f"Unknown column: {column_name}")
            
        classifier = self.classifiers[column_name]
        return await self._process_batches(texts, classifier, batch_size)
    
    async def _process_batches(self, texts: List[str], classifier_func, batch_size: int) -> List[Dict]:
        """배치 처리 공통 로직"""
        results = []
        texts = [text for text in texts if pd.notna(text) and str(text).strip()]
        
        if not texts:
            return []
            
        batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
        
        for batch in tqdm_asyncio(batches, desc=f"Processing {classifier_func.__name__}"):
            try:
                async with self.semaphore:
                    batch_results = await classifier_func(batch, self.semaphore)
                    if batch_results:
                        results.extend(batch_results)
            except Exception as e:
                print(f"Error processing batch: {str(e)}")
                continue
                
        return results

    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """API 호출 공통 로직"""
        try:
            response = await asyncio.to_thread(
                self.client.messages.create,
                model="claude-3-5-haiku-20241022",
                max_tokens=4096,
                temperature=0,
                system="JSON 형식으로 응답하세요.",
                messages=[{"role": "user", "content": prompt}]
            )
            
            content = response.content[0].text
            print(response.content)
            # JSON 배열 추출을 위한 강건한 접근
            def extract_json_array(text: str) -> List[Dict]:
                # 1. 가장 바깥쪽 대괄호([])와 그 내용을 찾음
                array_pattern = r'\[(?:[^[\]]*|\[(?:[^[\]]*|\[[^[\]]*\])*\])*\]'
                matches = list(re.finditer(array_pattern, text))
                
                if not matches:
                    return []
                    
                # 가장 긴 매치를 선택 (보통 완전한 JSON 배열이 가장 김)
                longest_match = max(matches, key=lambda match: len(match.group()))
                potential_json = longest_match.group()
                
                try:
                    # JSON 파싱 시도
                    parsed = json.loads(potential_json)
                    if isinstance(parsed, list):
                        return parsed
                    return []
                except json.JSONDecodeError:
                    return []
                    
            # JSON 추출 시도
            result = extract_json_array(content)
            
            if result:
                return result
            else:
                print("Failed to extract valid JSON array. Raw content:")
                print(content)
                return []
                
        except Exception as e:
            print(f"API call error: {str(e)}")
            if 'response' in locals():
                print(f"Raw response content: {response.content}")
            return []

    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """CC(Chief Complaint) 분류"""
        prompt = f"""다음 주요 증상(CC) 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. location: 통증/증상 위치 (문자열)
            2. pain_type: 통증/증상 종류 (문자열)
            3. severity: 통증/증상 강도 (1-5, 없으면 null)
            4. duration: 지속 기간 (명시된 경우만, 문자열)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "location": "위치",
                "pain_type": "통증 종류",
                "severity": 숫자 또는 null,
                "duration": "기간 또는 null"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        prompt = f"""다음 약물 복용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. medication_type: 약물 종류 (진통제/소염제/근이완제 등)
            2. frequency: 복용 빈도 ('regular': 정기적, 'occasional': 간헐적, 'none': 미복용)
            3. duration: 복용 기간 (명시된 경우만)
            4. compliance: 복약 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "medication_type": "약물 종류",
                "frequency": "복용 빈도",
                "duration": "기간 또는 null",
                "compliance": "순응도"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        prompt = f"""다음 장치 사용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. device_type: 장치 종류
            2. usage_pattern: 사용 패턴 ('constant': 상시착용, 'partial': 부분착용, 'rare': 거의미착용)
            3. duration: 사용 기간
            4. compliance: 착용 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "device_type": "장치 종류",
                "usage_pattern": "사용 패턴",
                "duration": "기간 또는 null",
                "compliance": "순응도"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        prompt = f"""다음 습관 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. habit_type: 습관 종류 (이갈이/편측성저작 등)
            2. frequency: 빈도 ('high': 매일/자주, 'medium': 가끔, 'low': 거의없음)
            3. awareness: 인지여부 ('aware': 인지, 'unaware': 미인지)
            4. improvement: 개선여부 ('improved': 개선, 'unchanged': 유지, 'worsened': 악화)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "habit_type": "습관 종류",
                "frequency": "빈도",
                "awareness": "인지여부",
                "improvement": "개선여부"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        prompt = f"""다음 찜질 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. status: 찜질 시행 여부 (0: 미시행, 1: 시행)
            2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
            3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
            4. method: 찜질 방법 ('hot': 온찜질, 'cold': 냉찜질, 'both': 둘 다, null: 불명확)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "status": 0 또는 1,
                "frequency": "빈도",
                "duration": 숫자 또는 null,
                "method": "방법"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지/스트레칭 분류"""
        prompt = f"""다음 마사지/스트레칭 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. type: 종류 ('massage': 마사지, 'stretching': 스트레칭, 'both': 둘다)
            2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
            3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
            4. method: 방법 ('self': 자가, 'professional': 전문가, 'both': 둘다)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "type": "종류",
                "frequency": "빈도",
                "duration": 숫자 또는 null,
                "method": "방법"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """현재 질환(PI) 분류"""
        prompt = f"""다음 현재 질환(PI) 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. onset: 증상 발현 시기
            2. pattern: 증상 양상 ('constant': 지속성, 'intermittent': 간헐성, 'progressive': 진행성)
            3. aggravating_factors: 악화 요인 (리스트)
            4. status: 현재 상태 ('improving': 호전중, 'unchanged': 유지, 'worsening': 악화)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "text": "원본 텍스트",
                "onset": "발현 시기",
                "pattern": "증상 양상",
                "aggravating_factors": ["요인1", "요인2"],
                "status": "현재 상태"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """의료 데이터 전체 처리"""
    classifier = MedicalTextClassifier(api_key)
    return await classifier.process_all_columns(df)

if __name__ == "__main__":
    # 데이터 처리
    loop = asyncio.get_event_loop()
    # df = pd.read_csv('test.csv', encoding='utf-8')
    processed_df = loop.run_until_complete(process_medical_data(df_sample, api_keys))
    
    # 결과 저장
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f'processed_medical_data_{timestamp}.csv'
    processed_df.to_csv(output_file, encoding='utf-8', index=False)
    
    # 기본 통계 출력
    print("\n=== 처리 결과 통계 ===")
    for column in processed_df.columns:
        if column.endswith(('_status', '_frequency', '_type', 'method')):
            print(f"\n{column} 분포:")
            print(processed_df[column].value_counts(dropna=False))

In [ ]:
processed_df['CC'].loc[16159]

In [ ]:
processed_df[['CC','CC_text', 'CC_location',
       'CC_pain_type', 'CC_severity', 'CC_duration']]

In [ ]:
processed_df[['약','약_text',
       '약_medication_type', '약_frequency', '약_duration', '약_compliance',]]

In [ ]:
processed_df[['장치',
       '장치_text', '장치_device_type', '장치_usage_pattern', '장치_duration','장치_compliance',
    #    '찜질_text', '찜질_status', '찜질_frequency', '찜질_duration','찜질_method', 
    #    '마사지, 스트레칭_text', '마사지, 스트레칭_type', '마사지, 스트레칭_frequency','마사지, 스트레칭_duration', '마사지, 스트레칭_method', 
    #    'PI_text', 'PI_onset','PI_pattern', 'PI_aggravating_factors', 'PI_status'
       ]]

In [ ]:
processed_df[['찜질',
    #    '장치_text', '장치_device_type', '장치_usage_pattern', '장치_duration','장치_compliance',
       '찜질_text', '찜질_status', '찜질_frequency', '찜질_duration','찜질_method', 
    #    '마사지, 스트레칭_text', '마사지, 스트레칭_type', '마사지, 스트레칭_frequency','마사지, 스트레칭_duration', '마사지, 스트레칭_method', 
    #    'PI_text', 'PI_onset','PI_pattern', 'PI_aggravating_factors', 'PI_status'
       ]]

In [ ]:
processed_df[['마사지, 스트레칭',
    #    '장치_text', '장치_device_type', '장치_usage_pattern', '장치_duration','장치_compliance',
    #    '찜질_text', '찜질_status', '찜질_frequency', '찜질_duration','찜질_method', 
       '마사지, 스트레칭_text', '마사지, 스트레칭_type', '마사지, 스트레칭_frequency','마사지, 스트레칭_duration', '마사지, 스트레칭_method', 
    #    'PI_text', 'PI_onset','PI_pattern', 'PI_aggravating_factors', 'PI_status'
       ]]

In [ ]:
processed_df[['PI',
    #    '장치_text', '장치_device_type', '장치_usage_pattern', '장치_duration','장치_compliance',
    #    '찜질_text', '찜질_status', '찜질_frequency', '찜질_duration','찜질_method', 
    #    '마사지, 스트레칭_text', '마사지, 스트레칭_type', '마사지, 스트레칭_frequency','마사지, 스트레칭_duration', '마사지, 스트레칭_method', 
       'PI_text', 'PI_onset','PI_pattern', 'PI_aggravating_factors', 'PI_status'
       ]] 

### konlpy

In [ ]:
from konlpy.tag import Mecab
mecab = Mecab()

In [ ]:
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
from collections import defaultdict
import numpy as np
from gensim.models import Word2Vec

class TextAnalyzer:
    def __init__(self):
        self.mecab = Mecab()
        self.vectorizer = TfidfVectorizer()
        
    def preprocess_texts(self, texts):
        processed_texts = []
        for text in texts:
            # 형태소 분석 - 명사, 동사, 형용사 추출
            morphs = self.mecab.pos(text)
            words = [word for word, pos in morphs 
                    if pos.startswith('N') or pos.startswith('V') or pos.startswith('MA')]
            processed_texts.append(words)
        return processed_texts
    
    def extract_key_terms(self, processed_texts, min_freq=5):
        # 단어 빈도 계산
        word_freq = defaultdict(int)
        for text in processed_texts:
            for word in set(text):  # 문서별 중복 제거
                word_freq[word] += 1
        
        # 최소 빈도 이상 등장한 단어만 선택
        key_terms = [word for word, freq in word_freq.items() if freq >= min_freq]
        return key_terms, word_freq
    
    def build_word_vectors(self, processed_texts):
        # Word2Vec 모델 학습
        model = Word2Vec(sentences=processed_texts, vector_size=100, window=5, min_count=1)
        return model
    
    def cluster_terms(self, word_vectors, key_terms, eps=0.5, min_samples=3):
        # 단어 벡터 추출
        vectors = np.array([word_vectors.wv[word] for word in key_terms])
        
        # DBSCAN 클러스터링
        clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(vectors)
        
        # 클러스터별 단어 그룹화
        clusters = defaultdict(list)
        for term, label in zip(key_terms, clustering.labels_):
            clusters[label].append(term)
            
        return clusters
    
    def analyze_categories(self, texts):
        # 텍스트 전처리
        processed_texts = self.preprocess_texts(texts)
        
        # 주요 용어 추출
        key_terms, word_freq = self.extract_key_terms(processed_texts)
        
        # 단어 벡터 학습
        word_vectors = self.build_word_vectors(processed_texts)
        
        # 용어 클러스터링
        clusters = self.cluster_terms(word_vectors, key_terms)
        
        # 결과 정리
        categories = {
            f"카테고리_{i}": {
                "terms": terms,
                "frequent_terms": sorted(
                    [(term, word_freq[term]) for term in terms],
                    key=lambda x: x[1],
                    reverse=True
                )[:5]
            }
            for i, terms in clusters.items()
            if i != -1  # 노이즈 제외
        }
        
        return categories

# 사용 예시
if __name__ == "__main__":
    analyzer = TextAnalyzer()
    
    # 샘플 데이터
    texts = [
        "샘플 텍스트 1",
        "샘플 텍스트 2",
        "샘플 텍스트 3"
    ]
    
    # 분석 실행
    categories = analyzer.analyze_categories(texts)
    
    # 결과 출력
    for category, info in categories.items():
        print(f"\n{category}:")
        print("상위 빈출 단어:")
        for term, freq in info['frequent_terms']:
            print(f"  - {term}: {freq}회")

In [ ]:
df.CC